# Proyecto: Lending Club
 
## Limpiezas:
   * term       " 36 months"  -> 36          (entero)
   * int_rate   "13.56%"       -> 13.56       (numero)
   * revol_util "52.3%"        -> 52.3        (numero)
   * issue_d    "Dec-2015"     -> 2015-12-01  (fecha con formato)
   * columnas de conteo -> entero que admite nulos (Int64)


In [ ]:
import pandas as pd
# Rutas de entrada y salida:
ENTRADA = r"C:\Users\Usuario\Documents\Lending Club Proyect\archive\loan.csv"
SALIDA  = r"C:\Users\Usuario\Documents\Lending Club Proyect\archive\loan_clean.csv"

In [ ]:
# Las 29 columnas elegidas
COLUMNAS = [
    # prestamo
    "loan_amnt", "funded_amnt", "term", "int_rate", "installment",
    "grade", "sub_grade", "issue_d", "purpose", "loan_status",
    # solicitante
    "emp_length", "home_ownership", "annual_inc", "verification_status",
    "dti", "addr_state",
    # perfil de credito
    "delinq_2yrs", "open_acc", "pub_rec", "revol_bal", "revol_util",
    "total_acc", "mort_acc", "pub_rec_bankruptcies",
    # dinero / resultado
    "out_prncp", "total_pymnt", "total_rec_prncp", "total_rec_int",
    "recoveries",
]


In [ ]:
# Columnas de conteo: enteros que admiten nulos (evita el "0.0")
COLS_ENTERAS = [
    "delinq_2yrs", "open_acc", "pub_rec", "total_acc",
    "mort_acc", "pub_rec_bankruptcies",
]
#Limpiar varios datos 
def limpiar(chunk: pd.DataFrame) -> pd.DataFrame:
    # term: " 36 months" a 36
    chunk["term"] = (
        chunk["term"].astype(str).str.extract(r"(\d+)")[0].astype("Int64")
    )

    # int_rate y revol_util: quitar '%' y pasar a numero
    for col in ["int_rate", "revol_util"]:
        chunk[col] = (
            chunk[col].astype(str).str.replace("%", "", regex=False).str.strip()
        )
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce")

    # issue_d: de texto a formato fecha 
    chunk["issue_d"] = pd.to_datetime(
        chunk["issue_d"], format="%b-%Y", errors="coerce"
    )

    # conteos entero nullable
    for col in COLS_ENTERAS:
        chunk[col] = pd.to_numeric(chunk[col], errors="coerce").astype("Int64")

    return chunk
primero = True
filas = 0

for chunk in pd.read_csv(
    ENTRADA, usecols=COLUMNAS, chunksize=200_000, low_memory=False
):
    chunk = chunk[COLUMNAS]          
    chunk = limpiar(chunk)
    chunk.to_csv(
        SALIDA,
        index=False,
        mode="w" if primero else "a",
        header=primero,
    )
    filas += len(chunk)
    primero = False
    print(f"Procesadas {filas:,} filas...")

print(f"\nListo. {filas:,} filas escritas en loan_clean.csv")
